# Amharic → English NLLB Fine-tuning (Dual-GPU, DDP)

This version uses **both T4 GPUs on Kaggle** via proper `DistributedDataParallel`
(launched with `torchrun`), instead of the naive `DataParallel` wrapping that
caused the earlier `ReduceAddCoalesced` OOM crash.

Because DDP requires a real multi-process launch, the training step runs as
a standalone script (`train.py`) launched with `torchrun --nproc_per_node=2`,
not as a normal notebook cell. Data cleaning, dedup, and splitting still run
normally in the notebook first.

**Before running:** set the Kaggle accelerator to **GPU T4 x2** in the
Settings panel on the right.

**Fix included in this version:** pins `transformers==4.57.1`. Newer
`transformers` v5.x has a regression where NLLB's tied embeddings
(`model.shared` / `encoder.embed_tokens` / `decoder.embed_tokens` /
`lm_head`) get loaded as separate, untied copies instead of one shared
matrix. With NLLB's ~256k-token vocab, each copy is ~1GB, so that bug alone
can add several extra GB of params + gradients + optimizer state — enough to
OOM a 14.56GB T4 even with a small batch size, fp16, and gradient
checkpointing already on. The training script also force-ties the
embeddings as a belt-and-suspenders fallback in case the pin can't be
applied in your environment.

## Step 1 — Load and clean the parallel corpus

Loads the six domain files (`amharic.txt` / `english.txt`), applies:
- Unicode NFC normalization
- Amharic homophone-character normalization (ሃ/ሐ/ሓ/ኀ/ኃ/ኸ → ሀ, ሠ/ሧ → ሰ/ሶ, ዐ/ዓ → አ)
- Curly-quote → straight-quote normalization
- Trailing double-period cleanup (`"foo.."` → `"foo."`)

In [1]:
import os
import re
import unicodedata
import pandas as pd

BASE = "/kaggle/input/datasets/fenetitefera/amharic-english1/amharic-english"
domains = [
    "amharic_english_ethiopiac_bible",
    "amharic_english_history",
    "amharic_english_Jw_bible",
    "amharic_english_JW_daily_quote",
    "amharic_english_legal",
    "amharic_english_news",
]

def normalize_quotes(text):
    return (text.replace("\u201c", '"').replace("\u201d", '"')
                .replace("\u2018", "'").replace("\u2019", "'"))

def clean_trailing(text):
    text = text.strip()
    text = re.sub(r"\s*\.\s*\.\s*$", ".", text)
    text = re.sub(r"\.{2,}$", ".", text)
    return text

def normalize_amharic(text):
    text = unicodedata.normalize("NFC", text)
    replacements = {
        "ሃ": "ሀ", "ሐ": "ሀ", "ሓ": "ሀ", "ኀ": "ሀ", "ኃ": "ሀ", "ኸ": "ሀ",
        "ሠ": "ሰ", "ሧ": "ሶ", "ዐ": "አ", "ዓ": "አ",
    }
    for k, v in replacements.items():
        text = text.replace(k, v)
    text = normalize_quotes(text)
    return clean_trailing(text)

def clean_english(text):
    text = unicodedata.normalize("NFC", text)
    text = re.sub(r"\s+", " ", text)
    text = normalize_quotes(text)
    return clean_trailing(text)

all_pairs = []
for d in domains:
    am_path = os.path.join(BASE, d, "amharic.txt")
    en_path = os.path.join(BASE, d, "english.txt")
    with open(am_path, encoding="utf-8") as f:
        am_lines = [l.strip() for l in f.readlines()]
    with open(en_path, encoding="utf-8") as f:
        en_lines = [l.strip() for l in f.readlines()]

    n = min(len(am_lines), len(en_lines))
    for am, en in zip(am_lines[:n], en_lines[:n]):
        am_c, en_c = normalize_amharic(am), clean_english(en)
        if am_c and en_c:
            all_pairs.append({"domain": d, "am": am_c, "en": en_c})

print(f"Loaded {len(all_pairs)} raw pairs across {len(domains)} domains")

Loaded 107928 raw pairs across 6 domains


## Step 2 — Deduplicate and drop extreme length-ratio outliers

In [2]:
df = pd.DataFrame(all_pairs).drop_duplicates(subset=["am", "en"])

df["am_wc"] = df["am"].str.split().str.len()
df["en_wc"] = df["en"].str.split().str.len()
df["len_ratio"] = df["en_wc"] / df["am_wc"].replace(0, 1)
before = len(df)
df = df[(df["len_ratio"] > 0.15) & (df["len_ratio"] < 6.0)]

print(f"After dedup + ratio filter: {len(df)} pairs (dropped {before - len(df)} outliers)")
print(df["domain"].value_counts())

After dedup + ratio filter: 64823 pairs (dropped 581 outliers)
domain
amharic_english_Jw_bible           30688
amharic_english_news               12506
amharic_english_ethiopiac_bible    11292
amharic_english_JW_daily_quote      4675
amharic_english_legal               4457
amharic_english_history             1205
Name: count, dtype: int64


## Step 3 — Train / validation / test split, save to CSV

Saved to disk because the training script (Step 6) runs as a **separate
process** via `torchrun` and can't see the notebook's in-memory variables —
it reloads these CSVs itself.

In [3]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(df, test_size=0.1, stratify=df["domain"], random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, stratify=temp_df["domain"], random_state=42)

print("Train:", len(train_df), "| Val:", len(val_df), "| Test:", len(test_df))

train_df.to_csv("/kaggle/working/train.csv", index=False)
val_df.to_csv("/kaggle/working/val.csv", index=False)
test_df.to_csv("/kaggle/working/test.csv", index=False)
print("Saved train.csv, val.csv, test.csv to /kaggle/working/")

Train: 58340 | Val: 3241 | Test: 3242
Saved train.csv, val.csv, test.csv to /kaggle/working/


## Step 4 — Install extras (and pin `transformers`)

**Pinning `transformers==4.57.1`** here fixes the tied-embeddings bug in
v5.x described above — without this pin, NLLB's embedding matrices load as
3-4 separate untied copies and can OOM even a correctly-configured dual-GPU
run. Restart the kernel after this cell if you had already imported
`transformers` earlier in the session, so the pinned version actually takes
effect.

In [4]:
!pip install -q "transformers==4.57.1" accelerate evaluate sacrebleu

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 696.9 kB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 60.9 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 4.0 MB/s eta 0:00:00


## Step 5 — Write the training script

Everything here — tokenizing, the corrected `compute_loss` (fixes the
gradient-accumulation loss-scaling bug from before), the checkpoint cleanup
callback, and training itself — lives inside one script so `torchrun` can
launch it once per GPU.

Key differences from the single-GPU version:
- `ddp_find_unused_parameters=False` for efficiency.
- `gradient_accumulation_steps` halved (16 → 8) since 2 GPUs already double
  the parallel batch size, keeping the same effective batch (~32).
- `trainer.is_world_process_zero()` guards the final save so only **one**
  of the two processes writes it.
- After loading the model, we force-tie the embedding/lm_head weights
  (`model.tie_weights()`) as a fallback in case the `transformers` pin
  above can't be applied in your environment — this avoids the 3-4x
  embedding-memory blowup that caused the earlier OOM.

In [32]:
%%writefile /kaggle/working/train.py
import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
import os
import shutil
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from datasets import Dataset
from transformers import (
    AutoTokenizer, AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq, Seq2SeqTrainingArguments, Seq2SeqTrainer, TrainerCallback,
)
import evaluate

MODEL = "facebook/nllb-200-distilled-600M"

train_df = pd.read_csv("/kaggle/working/train.csv")
val_df = pd.read_csv("/kaggle/working/val.csv")

tokenizer = AutoTokenizer.from_pretrained(MODEL, src_lang="amh_Ethi", tgt_lang="eng_Latn")
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL)

# --- Fix for the tied-embeddings OOM bug ---
# transformers v5.x sometimes refuses to tie model.shared / encoder.embed_tokens /
# decoder.embed_tokens / lm_head even though they hold identical weights, loading
# 3-4 separate ~1GB copies of NLLB's 256k-vocab embedding matrix instead of one.
# Force them back together to reclaim that memory (safe: NLLB is trained tied).
model.config.tie_word_embeddings = True
model.tie_weights()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
# --------------------------------------------

lang_id_src = tokenizer.convert_tokens_to_ids("amh_Ethi")
lang_id_tgt = tokenizer.convert_tokens_to_ids("eng_Latn")
unk_id = tokenizer.unk_token_id
MAX_LEN = 128

def fix_lang_tag_batch(id_lists, correct_lang_id):
    fixed = []
    for ids in id_lists:
        if ids[-1] == unk_id:
            ids = ids[:-1] + [correct_lang_id]
        fixed.append(ids)
    return fixed

def preprocess(batch):
    model_inputs = tokenizer(
        batch["am"], text_target=batch["en"], max_length=MAX_LEN, truncation=True
    )
    model_inputs["input_ids"] = fix_lang_tag_batch(model_inputs["input_ids"], lang_id_src)
    model_inputs["labels"] = fix_lang_tag_batch(model_inputs["labels"], lang_id_tgt)
    return model_inputs

train_ds = Dataset.from_pandas(train_df[["am", "en"]].reset_index(drop=True)).map(
    preprocess, batched=True, remove_columns=["am", "en"]
)
val_ds = Dataset.from_pandas(val_df[["am", "en"]].reset_index(drop=True)).map(
    preprocess, batched=True, remove_columns=["am", "en"]
)

torch.cuda.empty_cache()
model.config.use_cache = False

collator = DataCollatorForSeq2Seq(tokenizer, model=model)
bleu = evaluate.load("sacrebleu")

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    preds = np.where(preds != -100, preds, tokenizer.pad_token_id)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    result = bleu.compute(predictions=decoded_preds, references=[[l] for l in decoded_labels])
    return {"bleu": result["score"]}

args = Seq2SeqTrainingArguments(
    output_dir="/kaggle/working/nllb-am-en",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    eval_accumulation_steps=1,
    num_train_epochs=5,
    predict_with_generate=True,
    fp16=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    optim="adafactor",
    logging_steps=50,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="bleu",
    report_to="none",
    ddp_find_unused_parameters=False,
)

class FixedSeq2SeqTrainer(Seq2SeqTrainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs["labels"]
        outputs = model(**inputs)
        logits = outputs.logits
        loss = F.cross_entropy(
            logits.view(-1, logits.size(-1)), labels.view(-1),
            ignore_index=-100, reduction="sum",
        )
        denom = num_items_in_batch if num_items_in_batch is not None else (labels != -100).sum()
        return (loss / denom, outputs) if return_outputs else loss / denom

class CleanupCallback(TrainerCallback):
    def on_save(self, args, state, control, **kwargs):
        if state.is_world_process_zero and round(state.epoch) == 3:
            for folder in [
                "/kaggle/working/nllb-am-en/checkpoint-1824",
                "/kaggle/working/nllb-am-en/checkpoint-3648",
            ]:
                if os.path.exists(folder):
                    try:
                        shutil.rmtree(folder)
                        print(f"Deleted {folder} to free space")
                    except FileNotFoundError:
                        pass
        return control

trainer = FixedSeq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tokenizer,
    data_collator=collator,
    compute_metrics=compute_metrics,
    callbacks=[CleanupCallback()],
)

trainer.train(resume_from_checkpoint="/kaggle/working/nllb-am-en/checkpoint-1824")

if trainer.is_world_process_zero():
    final_dir = "/kaggle/working/nllb-am-en-final"
    trainer.save_model(final_dir)
    tokenizer.save_pretrained(final_dir)
    print("Saved final model to", final_dir)


Overwriting /kaggle/working/train.py


## Step 6 — Confirm both GPUs are visible, then launch training

If `nvidia-smi` doesn't show 2 GPUs, double check the accelerator setting is
**GPU T4 x2**, then restart the session.

To resume after an interruption: edit the `%%writefile` cell above, add
`resume_from_checkpoint=True` inside `trainer.train(...)`, re-run that cell
to rewrite the script, then re-run this launch cell.

In [6]:
!nvidia-smi --query-gpu=name,memory.total,memory.used,memory.free --format=csv

name, memory.total [MiB], memory.used [MiB], memory.free [MiB]
Tesla T4, 15360 MiB, 0 MiB, 14912 MiB
Tesla T4, 15360 MiB, 0 MiB, 14912 MiB


In [7]:
!nvidia-smi

Sat Sep 12 17:03:17 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   47C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

If a previous run left a zombie process holding GPU memory (visible in the
`nvidia-smi` output above under `Processes`), kill it by PID before
launching again — **replace `58` with the actual PID you see**:

In [8]:
# !kill -9 <PID>   # uncomment and set the real PID from nvidia-smi above, if needed

In [34]:
!torchrun --nproc_per_node=2 /kaggle/working/train.py

W0912 20:22:36.115000 331 torch/distributed/run.py:852] 
W0912 20:22:36.115000 331 torch/distributed/run.py:852] *****************************************
W0912 20:22:36.115000 331 torch/distributed/run.py:852] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0912 20:22:36.115000 331 torch/distributed/run.py:852] *****************************************
Map: 100%|█████████████████████████| 3241/3241 [00:00<00:00, 3987.66 examples/s]
There were missing keys in the checkpoint model loaded: ['model.encoder.embed_tokens.weight', 'model.decoder.embed_tokens.weight', 'lm_head.weight'].
There were missing keys in the checkpoint model loaded: ['model.encoder.embed_tokens.weight', 'model.decoder.embed_tokens.weight', 'lm_head.weight'].
{'loss': 1.2413, 'grad_norm': 1.0177592039108276, 'learning_rate': 1.5945175438596492e-05, 'epoch': 

## Step 7 — Zip the final model for download

In [ ]:
import os

ckpt_root = "/kaggle/working/nllb-am-en"
checkpoints = [d for d in os.listdir(ckpt_root) if d.startswith("checkpoint-")]
latest = max(checkpoints, key=lambda x: int(x.split("-")[1]))
LATEST_CHECKPOINT = os.path.join(ckpt_root, latest)
print("Latest checkpoint:", LATEST_CHECKPOINT)

In [ ]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
import shutil
import kagglehub

SOURCE_DIR = "/kaggle/working/nllb-am-en-final"   # or LATEST_CHECKPOINT from above
CLEAN_DIR = "/kaggle/working/nllb-am-en-lastepoch"

model = AutoModelForSeq2SeqLM.from_pretrained(SOURCE_DIR)
tokenizer = AutoTokenizer.from_pretrained(SOURCE_DIR, src_lang="amh_Ethi", tgt_lang="eng_Latn")

model.save_pretrained(CLEAN_DIR)
tokenizer.save_pretrained(CLEAN_DIR)

shutil.make_archive(CLEAN_DIR, "zip", CLEAN_DIR)
print("Saved:", CLEAN_DIR + ".zip")

kagglehub.dataset_upload(
    handle="fenetitefera/nllb-am-en-epoch2",   # same dataset, new version
    local_dataset_dir=CLEAN_DIR
)

In [12]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
import shutil

CHECKPOINT = "/kaggle/working/nllb-am-en/checkpoint-1824"
CLEAN_DIR = "/kaggle/working/nllb-am-en-epoch1"

# Load the epoch-1 checkpoint (model weights only, no optimizer/scheduler state)
model = AutoModelForSeq2SeqLM.from_pretrained(CHECKPOINT)
tokenizer = AutoTokenizer.from_pretrained(CHECKPOINT, src_lang="amh_Ethi", tgt_lang="eng_Latn")

# Save a clean, inference-only copy
model.save_pretrained(CLEAN_DIR)
tokenizer.save_pretrained(CLEAN_DIR)

# Zip it for download
shutil.make_archive(CLEAN_DIR, "zip", CLEAN_DIR)
print("Saved:", CLEAN_DIR + ".zip")

Saved: /kaggle/working/nllb-am-en-epoch1.zip


## Step 8 — Quick sanity-check translation

In [16]:
import pandas as pd
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

test_df = pd.read_csv("/kaggle/working/test.csv")
final_dir = "/kaggle/working/nllb-am-en-epoch1"   # not nllb-am-en-final
tok = AutoTokenizer.from_pretrained(final_dir, src_lang="amh_Ethi", tgt_lang="eng_Latn")
mdl = AutoModelForSeq2SeqLM.from_pretrained(final_dir).to("cuda")
def translate(text):
    inputs = tok(text, return_tensors="pt", truncation=True, max_length=128).to("cuda")
    out = mdl.generate(
        **inputs,
        forced_bos_token_id=tok.convert_tokens_to_ids("eng_Latn"),
        max_length=128,
        num_beams=5,
    )
    return tok.batch_decode(out, skip_special_tokens=True)[0]

print(translate(test_df["am"].iloc[0]))
print("Reference:", test_df["en"].iloc[0])

The regions may issue laws necessary to implement this proclamation.
Reference: The regional states may issue laws necessary for the implementation of this Proclamation.


In [17]:
import kagglehub
kagglehub.dataset_upload(
    handle="fenetitefera/nllb-am-en-epoch1",
    local_dataset_dir="/kaggle/working/nllb-am-en-epoch1"
)

Uploading Dataset https://api.kaggle.com/datasets/fenetitefera/nllb-am-en-epoch1 ...
Starting upload for file /kaggle/working/nllb-am-en-epoch1/model.safetensors

Uploading: 100%|██████████| 2.46G/2.46G [00:29<00:00, 82.3MB/s]

Upload successful: /kaggle/working/nllb-am-en-epoch1/model.safetensors (2GB)
Starting upload for file /kaggle/working/nllb-am-en-epoch1/tokenizer_config.json



Uploading: 100%|██████████| 40.0k/40.0k [00:00<00:00, 97.8kB/s]

Upload successful: /kaggle/working/nllb-am-en-epoch1/tokenizer_config.json (39KB)
Starting upload for file /kaggle/working/nllb-am-en-epoch1/generation_config.json


...


Uploading: 100%|██████████| 837/837 [00:00<00:00, 2.29kB/s]

Upload successful: /kaggle/working/nllb-am-en-epoch1/config.json (837B)
Starting upload for file /kaggle/working/nllb-am-en-epoch1/tokenizer.json



Uploading: 100%|██████████| 32.2M/32.2M [00:00<00:00, 51.5MB/s]

Upload successful: /kaggle/working/nllb-am-en-epoch1/tokenizer.json (31MB)


Your dataset has been created.
Files are being processed...
See at: https://api.kaggle.com/datasets/fenetitefera/nllb-am-en-epoch1


In [21]:
def translate(text, max_length=128, num_beams=5):
    inputs = tok(text, return_tensors="pt", truncation=True, max_length=max_length).to("cuda")
    out = mdl.generate(
        **inputs,
        forced_bos_token_id=tok.convert_tokens_to_ids("eng_Latn"),
        max_length=max_length,
        num_beams=num_beams,
    )
    return tok.batch_decode(out, skip_special_tokens=True)[0]

# Try your own Amharic sentence here
print(translate("የኢትዮጵያ የልዩ ተሰጥኦ እና ብቃት ልማት ተቋም eጌት ተብሎ ይጠራል።?"))

Ethiopian Special Talent and Skills Development Institute is called EGATE.


In [26]:
sentences = [
    "እንዴት ነህ?",
    "አዲስ አበባ የኢትዮጵያ ዋና ከተማ ናት።",
    "ስሜ ፈጠነ ነው።",
]

for s in sentences:
    print(f"Amharic: {s}")
    print(f"English: {translate(s)}")
    print("-" * 60)

Amharic: እንዴት ነህ?
English: How are you?
------------------------------------------------------------
Amharic: አዲስ አበባ የኢትዮጵያ ዋና ከተማ ናት።
English: Addis Ababa is the capital of Ethiopia.
------------------------------------------------------------
Amharic: ስሜ ፈጠነ ነው።
English: My name is Rapid.
------------------------------------------------------------


In [33]:
import os
os.remove("/kaggle/working/nllb-am-en-epoch1.zip")
print("Deleted old zip")

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/working/nllb-am-en-epoch1.zip'

In [31]:
trainer.train(resume_from_checkpoint="/kaggle/working/nllb-am-en/checkpoint-1824")

NameError: name 'trainer' is not defined

## If you still hit CUDA OOM with dual-GPU DDP

Each GPU runs its own full copy of the model + optimizer state (DDP doesn't
shard), so per-GPU memory pressure is similar to the single-GPU run. With
the tied-embeddings fix above this should no longer happen, but if it does:

1. Lower `per_device_train_batch_size` to `1`, raise
   `gradient_accumulation_steps` to compensate.
2. Lower `MAX_LEN` (inside the lang-tag-fix section of the script) from
   `128` to `96`.
3. Confirm `gradient_checkpointing=True` and `fp16=True` are both still set.
4. Restart the session between attempts to clear fragmented CUDA memory.
5. Double check the `transformers==4.57.1` pin actually took effect (check
   `import transformers; print(transformers.__version__)` after Step 4) —
   if it's still v5.x, the kernel needs a restart for the pin to apply.